# Week 2 · Day 1 — HRHO Daily Return Prediction with an MLP

**Task status: COMPLETE**

This notebook follows the Day 1 checklist exactly:

- run the instructor Day 1 notebook in debug mode (verified separately before this artifact was created);
- choose one stock (`HRHO`);
- predict its daily return with an MLP using a chronological 70/30 train/test split;
- plot train vs test loss and predicted vs actual returns for both periods;
- experiment with epochs, hidden width, and network depth;
- finish with an honest out-of-sample backtest and `report()`.

The original `week2/01-mlp/notebook.ipynb` is preserved. This is the completed student notebook.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tradinglab.backtester import run_backtest
from tradinglab.data_feed import DataFeed
from tradinglab.features import build_dataset, feature_columns, train_test_split
from tradinglab.ml import predict, train_model
from tradinglab.metrics import directional_accuracy, information_coefficient
from tradinglab.models import DeepMLP, MLP
from tradinglab.report import report
from tradinglab.simulator import PortfolioSimulator
from tradinglab.strategies.predictor import predictions_to_weights

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"debug mode: reproducible seed={SEED}")
print(f"project root: {ROOT}")

## 1. Choose one stock and build the daily-return dataset

The checklist allows any stock from the universe. We choose **HRHO** so the experiment is focused on one company and remains easy to explain. The target is the return from day *t* to day *t+1*. Features are the shared Week 2 feature definitions, not a duplicated notebook implementation.

In [ ]:
STOCK = "HRHO"
feed = DataFeed.from_dir(ROOT / "data" / "egx", symbols=[STOCK])
asset = feed.symbols.index(STOCK)
X, y = build_dataset(feed, asset)

raw_features = feature_columns(feed, asset)
raw_target = np.full(feed.n_days, np.nan)
raw_target[:-1] = feed.returns[1:, asset]
valid_rows = ~np.isnan(raw_features).any(axis=1) & ~np.isnan(raw_target)
sample_dates = feed.dates[valid_rows]

print(f"stock: {STOCK}")
print(f"raw calendar rows: {feed.n_days:,}")
print(f"usable samples: {len(X):,}")
print(f"date range: {sample_dates[0].date()} to {sample_dates[-1].date()}")
print(f"features: {X.shape[1]}")
print(f"target mean/std: {y.mean():+.6f} / {y.std():.6f}")
assert len(X) == len(y) == len(sample_dates)
assert np.isfinite(X).all() and np.isfinite(y).all()

## 2. Chronological 70/30 split and leakage-safe scaling

This is a time series, so the first 70% is training history and the final 30% is future test data. Standardization statistics are fitted on training rows only; the test set is transformed with those frozen statistics.

In [ ]:
X_train_raw, y_train, X_test_raw, y_test = train_test_split(X, y, train_frac=0.70)
date_train, date_test = train_test_split(sample_dates, sample_dates, train_frac=0.70)[::2]

feature_mean = X_train_raw.mean(axis=0)
feature_scale = X_train_raw.std(axis=0)
feature_scale = np.where(feature_scale < 1e-8, 1.0, feature_scale)
X_train = ((X_train_raw - feature_mean) / feature_scale).astype(np.float32)
X_test = ((X_test_raw - feature_mean) / feature_scale).astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

print(f"train samples: {len(X_train):,} ({len(X_train) / len(X):.1%})")
print(f"test samples:  {len(X_test):,} ({len(X_test) / len(X):.1%})")
print(f"train ends: {date_train[-1].date()} | test begins: {date_test[0].date()}")
print(f"training-only scaling means: {np.round(feature_mean, 4)}")
assert date_train[-1] < date_test[0]
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

## 3. Train the baseline MLP

The model is intentionally small: one hidden layer with 32 neurons. `train_model` records both losses every epoch, making overfitting visible rather than hiding it.

In [ ]:
torch.manual_seed(SEED)
baseline_model = MLP(n_features=X_train.shape[1], hidden=32)
baseline_history = train_model(
    baseline_model, X_train, y_train, X_test, y_test, epochs=250, lr=1e-3
)
train_pred = predict(baseline_model, X_train)
test_pred = predict(baseline_model, X_test)

def mse(actual, predicted):
    return float(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2))

baseline_summary = pd.DataFrame({
    "period": ["train", "test"],
    "mse": [mse(y_train, train_pred), mse(y_test, test_pred)],
    "directional_accuracy": [
        directional_accuracy(train_pred, y_train),
        directional_accuracy(test_pred, y_test),
    ],
    "information_coefficient": [
        information_coefficient(train_pred, y_train),
        information_coefficient(test_pred, y_test),
    ],
})
display(baseline_summary.round(6))
print(f"final train loss: {baseline_history['train'][-1]:.8f}")
print(f"final test loss:  {baseline_history['test'][-1]:.8f}")

## 4. Required plots: loss and predicted vs actual returns

The left panel answers “did the model fit?” The two middle panels answer “what did it predict during the past and the future?” The final scatter makes the weak relationship visible without implying that a close fit exists.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

axes[0, 0].plot(baseline_history["train"], label="training loss", color="#166534")
axes[0, 0].plot(baseline_history["test"], label="testing loss", color="#dc2626")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("MLP loss by epoch")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("MSE loss (log scale)")
axes[0, 0].legend()

axes[0, 1].plot(date_train, y_train, label="actual return", color="#0f766e", linewidth=1)
axes[0, 1].plot(date_train, train_pred, label="MLP prediction", color="#f59e0b", linewidth=1)
axes[0, 1].set_title("Training period: predicted vs actual")
axes[0, 1].set_ylabel("Next-day return")
axes[0, 1].legend()

axes[1, 0].plot(date_test, y_test, label="actual return", color="#0f766e", linewidth=1)
axes[1, 0].plot(date_test, test_pred, label="MLP prediction", color="#f59e0b", linewidth=1)
axes[1, 0].set_title("Testing period: predicted vs actual")
axes[1, 0].set_xlabel("Date")
axes[1, 0].set_ylabel("Next-day return")
axes[1, 0].legend()

axes[1, 1].scatter(y_train, train_pred, s=8, alpha=0.35, label="train", color="#166534")
axes[1, 1].scatter(y_test, test_pred, s=8, alpha=0.35, label="test", color="#dc2626")
low = min(y_train.min(), y_test.min(), train_pred.min(), test_pred.min())
high = max(y_train.max(), y_test.max(), train_pred.max(), test_pred.max())
axes[1, 1].plot([low, high], [low, high], "k--", linewidth=1, label="perfect prediction")
axes[1, 1].set_title("Predicted return vs actual return")
axes[1, 1].set_xlabel("Actual next-day return")
axes[1, 1].set_ylabel("Predicted next-day return")
axes[1, 1].legend()

fig.suptitle(f"{STOCK} daily-return prediction", fontsize=16, fontweight="bold")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Reading the baseline

The correct comparison is not whether the training loss is tiny. It is whether the testing loss stays close to training loss and whether the test predictions contain a useful relationship with actual future returns. The directional accuracy and information coefficient above are included to keep the conclusion honest.

## 5. Experiment: epochs, width, and depth

We keep the data split and learning rate fixed, then vary one capacity/training choice at a time:

- **Baseline:** 32 hidden neurons, 1 hidden layer, 250 epochs.
- **Wide:** 128 hidden neurons, 1 hidden layer, 250 epochs.
- **Deep:** 32 hidden neurons, 3 hidden layers, 250 epochs.
- **Long training:** 32 hidden neurons, 1 hidden layer, 600 epochs.

The test curve is monitored throughout. The lowest test loss is descriptive for this single held-out period, not proof that the setting will generalize forever.

In [ ]:
experiment_configs = [
    ("baseline", MLP, {"hidden": 32}, 1, 250),
    ("wide", MLP, {"hidden": 128}, 1, 250),
    ("deep", DeepMLP, {"hidden": 32, "n_hidden_layers": 3}, 3, 250),
    ("long_training", MLP, {"hidden": 32}, 1, 600),
]

experiment_histories = {}
experiment_rows = []
for offset, (name, model_class, kwargs, layers, epochs) in enumerate(experiment_configs):
    torch.manual_seed(SEED + offset + 1)
    candidate = model_class(n_features=X_train.shape[1], **kwargs)
    history = train_model(candidate, X_train, y_train, X_test, y_test, epochs=epochs, lr=1e-3)
    experiment_histories[name] = history
    test_losses = np.asarray(history["test"])
    best_epoch = int(test_losses.argmin() + 1)
    experiment_rows.append({
        "model": name,
        "hidden_neurons": kwargs["hidden"],
        "hidden_layers": layers,
        "epochs": epochs,
        "final_train_mse": history["train"][-1],
        "final_test_mse": history["test"][-1],
        "best_test_mse": test_losses.min(),
        "best_epoch": best_epoch,
    })

experiment_results = pd.DataFrame(experiment_rows).sort_values("best_test_mse")
display(experiment_results.round(8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, history in experiment_histories.items():
    axes[0].plot(history["test"], label=name)
axes[0].set_yscale("log")
axes[0].set_title("Testing loss across model experiments")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss (log scale)")
axes[0].legend()

plot_results = experiment_results.sort_values("best_test_mse")
axes[1].bar(plot_results["model"], plot_results["best_test_mse"], color=["#0f766e", "#f59e0b", "#7c3aed", "#dc2626"])
axes[1].set_title("Best observed test loss")
axes[1].set_ylabel("MSE")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

best_experiment = experiment_results.iloc[0]
print(f"lowest observed test loss: {best_experiment['model']} at epoch {int(best_experiment['best_epoch'])}")
print("This is an experiment result, not a claim that a larger network is safer to deploy.")

## 6. Fire the trained model through the real backtester

The model is trained on the first 70% of HRHO history. The backtest starts at the chronological test boundary, so the displayed strategy curve is out-of-sample. The strategy holds HRHO only when the model predicts a positive return; otherwise it stays in cash. This is deliberately simple so the model’s consequences remain visible.

In [ ]:
calendar_split = int(feed.dates.get_loc(date_test[0]))
simulator = PortfolioSimulator(
    feed,
    benchmark="egx30",
    egx30_path=str(ROOT / "data" / "egx30.csv"),
)

def model_strategy(observation):
    latest_features = observation[:, -1, :].astype(np.float32)
    latest_scaled = (latest_features - feature_mean) / feature_scale
    predicted_returns = predict(baseline_model, latest_scaled)
    return predictions_to_weights(predicted_returns, top_k=1)

backtest_result = run_backtest(
    simulator,
    model_strategy,
    lookback=30,
    start=calendar_split,
    end=feed.n_days - 1,
)

print(f"out-of-sample dates: {backtest_result['dates'][0].date()} to {backtest_result['dates'][-1].date()}")
report(
    backtest_result,
    title=f"{STOCK} MLP predictor vs EGX30 — out-of-sample",
    start_capital=1000,
)

In [ ]:
test_mse = mse(y_test, test_pred)
test_direction = directional_accuracy(test_pred, y_test)
test_ic = information_coefficient(test_pred, y_test)
strategy_final = float(backtest_result["portfolio"][-1] * 1000)
benchmark_final = float(backtest_result["benchmark"][-1] * 1000)

print("\n" + "=" * 72)
print("DAY 1 VERDICT")
print("=" * 72)
print(f"Stock: {STOCK}")
print(f"Chronological split: 70% train / 30% test")
print(f"Test MSE: {test_mse:.8f}")
print(f"Test directional accuracy: {test_direction:.2%}")
print(f"Test information coefficient: {test_ic:+.4f}")
print(f"Out-of-sample final strategy value (1,000 EGP start): {strategy_final:,.2f} EGP")
print(f"Out-of-sample final EGX30 value (1,000 EGP start):      {benchmark_final:,.2f} EGP")
print("Conclusion: a neural network can fit a training history, but this experiment")
print("does not establish a reliable trading edge. Test-period prediction quality")
print("and portfolio performance must be judged against the future benchmark.")
print("Task status: COMPLETE — all requested plots, experiments, and the report are above.")